# 04 — Monte Carlo Knockout Stage Simulation

Retrain the XGBoost model on the **full** `matches_final_v2.csv` dataset (all 23,000+ matches
including 2026 group stage results), then run a 10,000-iteration Monte Carlo simulation
over the Round of 32 bracket to estimate each team's probability of reaching each stage.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
from collections import defaultdict
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

warnings.filterwarnings('ignore')
print("Imports OK")

## 1. Load Data

In [ ]:
matches = pd.read_csv('../data/processed/matches_final_v2.csv')
matches['date'] = pd.to_datetime(matches['date'])

rankings = pd.read_csv('../data/raw/fifa_ranking-2024-06-20.csv', parse_dates=['rank_date'])

features = ['weight', 'rank_diff', 'home_form', 'away_form', 'h2h']

print(f"Dataset shape: {matches.shape}")
print(f"Date range: {matches['date'].min().date()} -> {matches['date'].max().date()}")
print(f"Result distribution:\n{matches['result'].value_counts()}")

## 2. Retrain on Full Dataset

In [ ]:
# Train on full dataset — goal is best predictions, not evaluation
X_train = matches[features]
y_train = matches['result_encoded']

print(f"Training on {len(matches):,} matches (full dataset, no split)")

# compute sample weights to handle class imbalance
# this tells XGBoost to pay more attention to draws
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_model2 = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

# Note: no early_stopping_rounds since we have no holdout set — runs full 500 trees
xgb_model2.fit(
    X_train, y_train,
    sample_weight=sample_weights,  # handle class imbalance
    verbose=50
)

print(f"\nModel trained. Trees: {xgb_model2.n_estimators}")

joblib.dump(xgb_model2, '../models/xgb_model_v2.pkl')
joblib.dump(features, '../models/features.pkl')
print("Saved: models/xgb_model_v2.pkl")

## 3. Prediction Helper

In [ ]:
_RANK_NAME = {
    'Turkiye': 'Turkey',
    'IR Iran': 'Iran',
    'Korea Republic': 'Korea Republic',
}

def get_rank(team, date):
    t = _RANK_NAME.get(team, team)
    rows = rankings[(rankings['country_full'] == t) & (rankings['rank_date'] <= date)]
    if rows.empty:
        rows = rankings[
            (rankings['country_full'].str.lower() == t.lower()) &
            (rankings['rank_date'] <= date)
        ]
    return None if rows.empty else int(rows.iloc[-1]['rank'])

def get_form(team, date, n=10):
    past = matches[
        ((matches['home_team'] == team) | (matches['away_team'] == team)) &
        (matches['date'] < date)
    ].tail(n)
    if past.empty: return 0.5
    wins = sum(
        1 for _, r in past.iterrows()
        if (r['home_team'] == team and r['result'] == 'home_win') or
           (r['away_team'] == team and r['result'] == 'away_win')
    )
    return wins / len(past)

def get_h2h(home, away, date):
    h2h = matches[
        ((matches['home_team'] == home) & (matches['away_team'] == away)) &
        (matches['date'] < date)
    ]
    if h2h.empty: return 0.5
    return (h2h['result'] == 'home_win').sum() / len(h2h)

KNOCKOUT_DATE = pd.Timestamp('2026-06-28')

def predict_match_v2(home, away):
    rank_home = get_rank(home, KNOCKOUT_DATE)
    rank_away = get_rank(away, KNOCKOUT_DATE)
    if rank_home is None or rank_away is None:
        missing = home if rank_home is None else away
        print(f"  [WARN] No ranking for: {missing} — 50/50 fallback")
        return None
    rank_diff = rank_home - rank_away
    home_form = get_form(home, KNOCKOUT_DATE)
    away_form = get_form(away, KNOCKOUT_DATE)
    h2h       = get_h2h(home, away, KNOCKOUT_DATE)
    X = pd.DataFrame([[5, rank_diff, home_form, away_form, h2h]], columns=features)
    probs = xgb_model2.predict_proba(X)[0]  # [away_win, draw, home_win]
    return {
        'home': home, 'away': away,
        'prob_home_win': round(probs[2]*100, 1),
        'prob_draw':     round(probs[1]*100, 1),
        'prob_away_win': round(probs[0]*100, 1),
    }

# Quick test
r = predict_match_v2('Argentina', 'France')
print(f"Argentina vs France: {r['prob_home_win']}% / {r['prob_draw']}% / {r['prob_away_win']}%")

## 4. Bracket Definition

In [ ]:
R32 = [
    ('South Africa',            'Canada',                  'R32_01'),
    ('Netherlands',             'Morocco',                 'R32_02'),
    ('Germany',                 'Paraguay',                'R32_03'),
    ('France',                  'Sweden',                  'R32_04'),
    ('Belgium',                 'Senegal',                 'R32_05'),
    ('USA',                     'Bosnia and Herzegovina',  'R32_06'),
    ('Spain',                   'Austria',                 'R32_07'),
    ('Portugal',                'Croatia',                 'R32_08'),
    ('Brazil',                  'Japan',                   'R32_09'),
    ("Côte d'Ivoire",           'Norway',                  'R32_10'),
    ('Mexico',                  'Ecuador',                 'R32_11'),
    ('England',                 'Congo DR',                'R32_12'),
    ('Switzerland',             'Algeria',                 'R32_13'),
    ('Colombia',                'Ghana',                   'R32_14'),
    ('Australia',               'Egypt',                   'R32_15'),
    ('Argentina',               'Cabo Verde',              'R32_16'),
]

R16_PAIRS = [
    ('R32_01', 'R32_02', 'R16_01'),
    ('R32_03', 'R32_04', 'R16_02'),
    ('R32_05', 'R32_06', 'R16_03'),
    ('R32_07', 'R32_08', 'R16_04'),
    ('R32_09', 'R32_10', 'R16_05'),
    ('R32_11', 'R32_12', 'R16_06'),
    ('R32_13', 'R32_14', 'R16_07'),
    ('R32_15', 'R32_16', 'R16_08'),
]

QF_PAIRS = [
    ('R16_01', 'R16_02', 'QF_01'),
    ('R16_03', 'R16_04', 'QF_02'),
    ('R16_05', 'R16_06', 'QF_03'),
    ('R16_07', 'R16_08', 'QF_04'),
]

SF_PAIRS = [
    ('QF_01', 'QF_02', 'SF_01'),
    ('QF_03', 'QF_04', 'SF_02'),
]

print("R32 — model probabilities (home% / draw% / away%):")
for h, a, mid in R32:
    r = predict_match_v2(h, a)
    if r:
        print(f"  {mid}: {h} vs {a}  |  {r['prob_home_win']}% / {r['prob_draw']}% / {r['prob_away_win']}%")
    else:
        print(f"  {mid}: {h} vs {a}  |  [50/50 fallback]")

## 5. Monte Carlo Simulation

10,000 simulations through the full bracket.
No draws in knockout — draw probability redistributed proportionally between home/away win.


In [ ]:
# Precompute win probabilities for all possible pairings (max 32x31 = 992 ordered pairs)
# This means predict_match_v2 is called once per pair, not 320,000 times
print("Precomputing match probabilities...")
all_teams = [t for h, a, _ in R32 for t in (h, a)]
prob_cache = {}

for home in all_teams:
    for away in all_teams:
        if home == away:
            continue
        r = predict_match_v2(home, away)
        if r is None:
            prob_cache[(home, away)] = 0.5  # 50/50 fallback
        else:
            ph  = r['prob_home_win'] / 100
            pa  = r['prob_away_win'] / 100
            pd_ = r['prob_draw'] / 100
            total  = ph + pa
            ph_adj = ph + pd_ * (ph / total)
            prob_cache[(home, away)] = ph_adj

print(f"Cached {len(prob_cache)} pairings. Ready to simulate.")

In [ ]:
def knockout_winner(home, away):
    ph_adj = prob_cache.get((home, away), 0.5)
    return home if np.random.random() < ph_adj else away


def play_round(pairs, winners, losers, stage_label, progress):
    round_winners = {}
    for mid_a, mid_b, mid_out in pairs:
        home = winners[mid_a]
        away = winners[mid_b]
        w = knockout_winner(home, away)
        l = away if w == home else home
        progress[w] = stage_label
        progress[l] = f'Eliminated_{stage_label}'
        round_winners[mid_out] = w
        losers[mid_out] = l
    return round_winners


def simulate_tournament():
    progress = {}
    winners  = {}
    losers   = {}

    # R32
    for home, away, mid in R32:
        w = knockout_winner(home, away)
        l = away if w == home else home
        winners[mid] = w
        losers[mid]  = l
        progress[w]  = 'R32'
        progress[l]  = 'Eliminated_R32'

    # R16
    winners.update(play_round(R16_PAIRS, winners, losers, 'R16', progress))

    # QF
    winners.update(play_round(QF_PAIRS, winners, losers, 'QF', progress))

    # SF
    winners.update(play_round(SF_PAIRS, winners, losers, 'SF', progress))

    # Third place playoff
    tp_w = knockout_winner(losers['SF_01'], losers['SF_02'])
    tp_l = losers['SF_02'] if tp_w == losers['SF_01'] else losers['SF_01']
    progress[tp_w] = 'Third'
    progress[tp_l] = 'Fourth'

    # Final
    champion  = knockout_winner(winners['SF_01'], winners['SF_02'])
    runner_up = winners['SF_02'] if champion == winners['SF_01'] else winners['SF_01']
    progress[champion]  = 'Winner'
    progress[runner_up] = 'Final'

    return progress


N = 10_000
np.random.seed(42)
print(f"Running {N:,} simulations...")

stage_counts = defaultdict(lambda: defaultdict(int))
all_teams = [t for h, a, _ in R32 for t in (h, a)]

for _ in range(N):
    for team, stage in simulate_tournament().items():
        stage_counts[team][stage] += 1

print("Done!")

## 6. Results

In [ ]:
# Final stage labels in stage_counts:
#   Eliminated_R32, Eliminated_R16, Eliminated_QF, Third, Fourth, Final, Winner
# Intermediate labels (R32, R16, QF, SF) get overwritten each round — never appear in final counts

rows = []
for team in all_teams:
    c = stage_counts[team]
    row = {'team': team}

    # How many sims did this team exit at each stage
    row['pct_exit_R32']  = round(c.get('Eliminated_R32', 0) / N * 100, 1)
    row['pct_exit_R16']  = round(c.get('Eliminated_R16', 0) / N * 100, 1)
    row['pct_exit_QF']   = round(c.get('Eliminated_QF',  0) / N * 100, 1)
    row['pct_Third']     = round(c.get('Third',   0) / N * 100, 1)
    row['pct_Fourth']    = round(c.get('Fourth',  0) / N * 100, 1)
    row['pct_Final']     = round(c.get('Final',   0) / N * 100, 1)
    row['pct_Winner']    = round(c.get('Winner',  0) / N * 100, 1)

    # Cumulative reach: % of sims where team survived past each round
    row['pct_reach_R16']   = round(sum(c.get(s, 0) for s in ['Eliminated_R16','Eliminated_QF','Third','Fourth','Final','Winner']) / N * 100, 1)
    row['pct_reach_QF']    = round(sum(c.get(s, 0) for s in ['Eliminated_QF','Third','Fourth','Final','Winner']) / N * 100, 1)
    row['pct_reach_SF']    = round(sum(c.get(s, 0) for s in ['Third','Fourth','Final','Winner']) / N * 100, 1)
    row['pct_reach_Final'] = round(sum(c.get(s, 0) for s in ['Final','Winner']) / N * 100, 1)
    row['pct_win_title']   = round(c.get('Winner', 0) / N * 100, 1)
    rows.append(row)

results_df = pd.DataFrame(rows).sort_values('pct_win_title', ascending=False).reset_index(drop=True)

print("World Cup Winner Probabilities (Top 10):")
print(results_df[['team','pct_reach_R16','pct_reach_QF','pct_reach_SF','pct_reach_Final','pct_win_title']].head(10).to_string(index=False))

In [ ]:
results_df.to_csv('../data/processed/monte_carlo_r32.csv', index=False)
print("Saved: data/processed/monte_carlo_r32.csv")
print(f"\nFull table ({results_df.shape[0]} teams):")
print(results_df[['team','pct_win_title','pct_reach_Final','pct_reach_SF','pct_reach_QF','pct_reach_R16']].to_string(index=False))